# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 6.3 — Random Forest Modeling

### Objective

Train and evaluate a Random Forest classifier using the leakage-safe feature set.

The goals of this phase are:

- Capture non-linear churn patterns
- Improve predictive performance beyond Logistic Regression
- Evaluate business-relevant metrics
- Analyze feature importance
- Compare against the baseline Logistic Regression model

---

### Selected Dataset

Model B

Retention-related features excluded:

- RetentionCalls
- RetentionOffersAccepted
- MadeCallToRetentionTeam

---

### Baseline Benchmark

Balanced Logistic Regression

ROC-AUC: 0.603

This serves as the benchmark for comparison.

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 1: Load Dataset

Load the feature-engineered dataset and recreate the final leakage-safe modeling dataset used in Phase 6.2.

In [2]:
# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("../data/processed/feature_engineered.csv")

print("Dataset Loaded")
print("Shape:", df.shape)

Dataset Loaded
Shape: (51047, 67)


C:\Users\Akshit\AppData\Local\Temp\ipykernel_5232\154370990.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/feature_engineered.csv")


In [3]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [4]:
# ==========================================
# REMOVE EXCLUDED FEATURES
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea",
    "MarketCode",
    "AreaCode",
    "CreditRating",
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

model_df = df.drop(
    columns=columns_to_drop
)

print("Shape:", model_df.shape)

Shape: (51047, 59)


In [5]:
# ==========================================
# FEATURES & TARGET
# ==========================================

X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (51047, 58)
y Shape: (51047,)


In [6]:
# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Split completed.")

Split completed.


In [7]:
# ==========================================
# FEATURE TYPES
# ==========================================

numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 38
Categorical: 20


In [8]:
# ==========================================
# PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessor ready.")

Preprocessor ready.


## Step 2: Build Random Forest Pipeline

Random Forest is an ensemble learning algorithm that combines multiple decision trees.

Advantages:

- Captures non-linear relationships
- Handles feature interactions
- Less sensitive to scaling
- Often performs well on churn prediction problems

In [10]:
# ==========================================
# RANDOM FOREST PIPELINE
# ==========================================

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest pipeline created.")

Random Forest pipeline created.


In [11]:
type(rf_pipeline)

sklearn.pipeline.Pipeline

In [14]:
rf_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
# ==========================================
# TRAIN RANDOM FOREST
# ==========================================

rf_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")

Random Forest training completed.


## Step 3: Generate Predictions

Use the trained Random Forest model to generate:

- Class predictions
- Churn probabilities

These outputs will be used for performance evaluation.

In [17]:
# ==========================================
# RANDOM FOREST PREDICTIONS
# ==========================================

y_pred_rf = rf_pipeline.predict(
    X_valid
)

y_prob_rf = rf_pipeline.predict_proba(
    X_valid
)[:, 1]

print("Predictions generated.")

Predictions generated.


## Step 4: Evaluate Random Forest Performance

Evaluate the model using:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

These metrics will be compared against the Logistic Regression baseline.

In [18]:
# ==========================================
# RANDOM FOREST METRICS
# ==========================================

accuracy_rf = accuracy_score(
    y_valid,
    y_pred_rf
)

precision_rf = precision_score(
    y_valid,
    y_pred_rf
)

recall_rf = recall_score(
    y_valid,
    y_pred_rf
)

f1_rf = f1_score(
    y_valid,
    y_pred_rf
)

roc_auc_rf = roc_auc_score(
    y_valid,
    y_prob_rf
)

print("Accuracy :", round(accuracy_rf, 4))
print("Precision:", round(precision_rf, 4))
print("Recall   :", round(recall_rf, 4))
print("F1 Score :", round(f1_rf, 4))
print("ROC AUC  :", round(roc_auc_rf, 4))

Accuracy : 0.7166
Precision: 0.5789
Recall   : 0.0598
F1 Score : 0.1084
ROC AUC  : 0.6591


In [19]:
# ==========================================
# CONFUSION MATRIX
# ==========================================

cm_rf = confusion_matrix(
    y_valid,
    y_pred_rf
)

pd.DataFrame(
    cm_rf,
    index=["Actual_No", "Actual_Yes"],
    columns=["Pred_No", "Pred_Yes"]
)

,Pred_No,Pred_Yes
Actual_No,7140,128
Actual_Yes,2766,176


## Step 5: Threshold Tuning

The default classification threshold for Random Forest is 0.50.

For churn prediction, lower thresholds are often evaluated because:

- Missing churners is costly
- Higher recall is usually preferred
- Different thresholds create different business trade-offs

The objective is to identify a threshold that balances:

- Precision
- Recall
- F1 Score

In [20]:
# ==========================================
# THRESHOLD TUNING
# ==========================================

thresholds = [0.50, 0.40, 0.35, 0.30, 0.25, 0.20]

results = []

for threshold in thresholds:

    y_pred_threshold = (
        y_prob_rf >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        y_pred_threshold
    )

    recall = recall_score(
        y_valid,
        y_pred_threshold
    )

    f1 = f1_score(
        y_valid,
        y_pred_threshold
    )

    results.append([
        threshold,
        round(precision, 4),
        round(recall, 4),
        round(f1, 4)
    ])

threshold_df = pd.DataFrame(
    results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

threshold_df

,Threshold,Precision,Recall,F1 Score
0,0.50,0.5697,0.0653,0.1171
1,0.40,0.4749,0.2699,0.3442
2,0.35,0.4201,0.4521,0.4355
3,0.30,0.3800,0.6519,0.4802
4,0.25,0.3505,0.8188,0.4908
5,0.20,0.3241,0.9174,0.4790


In [21]:
# Sort by best F1

threshold_df.sort_values(
    by="F1 Score",
    ascending=False
)

,Threshold,Precision,Recall,F1 Score
4,0.25,0.3505,0.8188,0.4908
3,0.30,0.3800,0.6519,0.4802
5,0.20,0.3241,0.9174,0.4790
2,0.35,0.4201,0.4521,0.4355
1,0.40,0.4749,0.2699,0.3442
0,0.50,0.5697,0.0653,0.1171


## Step 6: Feature Importance Analysis

Random Forest provides feature importance scores that indicate how much each feature contributes to churn prediction.

The objective is to:

- Identify key churn drivers
- Generate business insights
- Support retention strategies
- Improve model interpretability

In [22]:
# ==========================================
# EXTRACT TRAINED RANDOM FOREST
# ==========================================

rf_model = rf_pipeline.named_steps["model"]

print(type(rf_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [23]:
# ==========================================
# GET FEATURE NAMES AFTER ENCODING
# ==========================================

feature_names = rf_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Total Features After Encoding:")
print(len(feature_names))

Total Features After Encoding:
144


In [24]:
# ==========================================
# FEATURE IMPORTANCE
# ==========================================

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(20)

,Feature,Importance
132,remainder__CurrentEquipmentDays,0.049668
112,remainder__PercChangeMinutes,0.042074
107,remainder__MonthlyMinutes,0.041177
142,remainder__CustomerValueProxy,0.037894
106,remainder__MonthlyRevenue,0.036201
113,remainder__PercChangeRevenues,0.035971
127,remainder__MonthsInService,0.033912
122,remainder__PeakCallsInOut,0.032393
123,remainder__OffPeakCallsInOut,0.032111
116,remainder__UnansweredCalls,0.030823


In [25]:
"CustomerValueProxy" in model_df.columns

True

In [26]:
model_df.columns.tolist()

['Churn',
 'MonthlyRevenue',
 'MonthlyMinutes',
 'TotalRecurringCharge',
 'DirectorAssistedCalls',
 'OverageMinutes',
 'RoamingCalls',
 'PercChangeMinutes',
 'PercChangeRevenues',
 'DroppedCalls',
 'BlockedCalls',
 'UnansweredCalls',
 'CustomerCareCalls',
 'ThreewayCalls',
 'ReceivedCalls',
 'OutboundCalls',
 'InboundCalls',
 'PeakCallsInOut',
 'OffPeakCallsInOut',
 'DroppedBlockedCalls',
 'CallForwardingCalls',
 'CallWaitingCalls',
 'MonthsInService',
 'UniqueSubs',
 'ActiveSubs',
 'Handsets',
 'HandsetModels',
 'CurrentEquipmentDays',
 'AgeHH1',
 'AgeHH2',
 'ChildrenInHH',
 'HandsetRefurbished',
 'HandsetWebCapable',
 'TruckOwner',
 'RVOwner',
 'Homeownership',
 'BuysViaMailOrder',
 'RespondsToMailOffers',
 'OptOutMailings',
 'NonUSTravel',
 'OwnsComputer',
 'HasCreditCard',
 'NewCellphoneUser',
 'NotNewCellphoneUser',
 'ReferralsMadeBySubscriber',
 'IncomeGroup',
 'OwnsMotorcycle',
 'AdjustmentsToCreditRating',
 'HandsetPrice',
 'PrizmCode',
 'Occupation',
 'MaritalStatus',
 'AgeHH2

In [29]:
[col for col in model_df.columns if "CustomerValue" in col]

['CustomerValueProxy']

In [30]:
importance_df.head(30)

,Feature,Importance
132,remainder__CurrentEquipmentDays,0.049668
112,remainder__PercChangeMinutes,0.042074
107,remainder__MonthlyMinutes,0.041177
142,remainder__CustomerValueProxy,0.037894
106,remainder__MonthlyRevenue,0.036201
113,remainder__PercChangeRevenues,0.035971
127,remainder__MonthsInService,0.033912
122,remainder__PeakCallsInOut,0.032393
123,remainder__OffPeakCallsInOut,0.032111
116,remainder__UnansweredCalls,0.030823


In [31]:
model_df_no_proxy = model_df.drop(
    columns=["CustomerValueProxy"]
)